# 中国裁判文书网爬虫

本 notebook 演示如何使用 Selenium 自动化爬取中国裁判文书网的法律文书。

## 学习目标
- 自动登录网站
- 处理 iframe 框架
- 高级检索和筛选
- 批量下载文档
- 处理分页和动态加载

## 注意事项
⚠️ 使用前请确保：
1. 已在中国裁判文书网注册账号
2. 修改代码中的手机号和密码
3. 设置正确的下载路径
4. 遵守网站使用条款

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.select import Select
import time

## 1. 配置浏览器选项

In [ ]:
# 目标网站
url = 'https://wenshu.court.gov.cn/website/wenshu/181029CR4M5A62CH/index.html?'

# 配置 Chrome 选项
option = webdriver.ChromeOptions()
option.add_argument('--start-maximized')
option.add_experimental_option('excludeSwitches', ['enable-automation'])

# 配置下载选项
prefs = {
    'profile.default_content_settings.popups': 0,
    'download.default_directory': '../data',
    'profile.default_content_setting_values.automatic_downloads': 1
}
option.add_experimental_option('prefs', prefs)

print("浏览器选项配置完成")

## 2. 初始化浏览器并访问网站

In [ ]:
# 创建浏览器实例
driver = webdriver.Chrome(options=option)
driver.maximize_window()
driver.set_page_load_timeout(30)
driver.get(url)

print(f"已访问: {url}")

## 3. 自动登录

⚠️ **重要**: 请将下面的 `'phoneNumber'` 和 `'password'` 替换为你的真实账号信息

In [ ]:
# 点击登录按钮
driver.find_element(By.XPATH, '//*[@id="loginLi"]/a').click()
time.sleep(10)

# 切换到 iframe 框架
iframe = driver.find_elements(By.TAG_NAME, 'iframe')[0]
driver.switch_to.frame(iframe)

# 输入手机号
username = driver.find_element(By.XPATH, '//*[@id="root"]/div/form/div/div[1]/div/div/div/input')
username.send_keys('phoneNumber')  # 替换为你的手机号
time.sleep(3)

# 输入密码
password = driver.find_element(By.XPATH, '//*[@id="root"]/div/form/div/div[2]/div/div/div/input')
password.send_keys('password')  # 替换为你的密码
time.sleep(2)

# 点击登录
driver.find_element(By.XPATH, '//*[@id="root"]/div/form/div/div[3]/span').click()
time.sleep(3)

# 退出 iframe
driver.switch_to.default_content()

print("登录完成")

## 4. 选择案件类型

可选类型：刑事案件、民事案件、行政案件等

In [ ]:
# 选择案件类型（可修改为其他类型）
case_type = '刑事案件'  # 可改为：民事案件、行政案件等
driver.find_element(By.LINK_TEXT, case_type).click()
time.sleep(10)

# 切换到新窗口
_lastWindow = driver.window_handles[-1]
driver.switch_to.window(_lastWindow)

print(f"已选择案件类型: {case_type}")

## 5. 高级检索和筛选

In [ ]:
# 按法院层级和地域筛选
driver.find_element(By.LINK_TEXT, '高级法院(146202)').click()
time.sleep(3)
driver.find_element(By.LINK_TEXT, '上海市(1188)').click()
time.sleep(3)

# 输入关键词检索
keyword = driver.find_element(By.XPATH, '//*[@id="_view_1545034775000"]/div/div[1]/div[2]/input')
search_keyword = '工程纠纷'  # 可修改为其他关键词
keyword.send_keys(search_keyword)
time.sleep(3)

# 点击搜索
driver.find_element(By.XPATH, '//*[@id="_view_1545034775000"]/div/div[1]/div[3]').click()
time.sleep(5)

print(f"检索关键词: {search_keyword}")

## 6. 设置每页显示数量

In [ ]:
# 设置每页显示 15 条（最大值）
page_size_box = Select(driver.find_element(By.XPATH, '//*[@id="_view_1545184311000"]/div[8]/div/select'))
page_size_box.select_by_visible_text('15')
time.sleep(3)

print("已设置每页显示 15 条")

## 7. 批量下载文档

最多可下载 600 条文档（40 页 × 15 条）

In [ ]:
def test_exceptions(xpath):
    """测试元素是否存在"""
    try:
        driver.find_element(By.XPATH, xpath)
        return True
    except:
        return False

# 下载统计
total_downloaded = 0
max_pages = 40  # 最多 40 页

print(f"开始批量下载，最多 {max_pages} 页...")

In [ ]:
from tqdm.notebook import tqdm

# 遍历每一页
for page in tqdm(range(1, max_pages + 1), desc="下载进度"):
    time.sleep(5 + page/10)
    
    # 遍历当前页的每条记录
    for i in range(15):
        time.sleep(5 + i/10)
        
        # 尝试两种可能的下载按钮位置
        event_xpath = f'//*[@id="_view_1545184311000"]/div[{i+3}]/div[6]/div/a[2]'
        
        if test_exceptions(event_xpath):
            driver.find_element(By.XPATH, event_xpath).click()
            total_downloaded += 1
        else:
            event_xpath = f'//*[@id="_view_1545184311000"]/div[{i+3}]/div[5]/div/a[2]'
            if test_exceptions(event_xpath):
                driver.find_element(By.XPATH, event_xpath).click()
                total_downloaded += 1
    
    # 点击下一页
    time.sleep(5)
    try:
        driver.find_element(By.LINK_TEXT, '下一页').click()
        driver.switch_to.default_content()
    except:
        print(f"已到最后一页，第 {page} 页")
        break

print(f"\n下载完成！共下载 {total_downloaded} 个文档")

## 8. 关闭浏览器

In [ ]:
# 关闭浏览器
driver.quit()
print("浏览器已关闭")

## 使用技巧

### 修改检索条件

1. **案件类型**: 修改 `case_type` 变量
   - 刑事案件
   - 民事案件
   - 行政案件
   - 赔偿案件
   - 执行案件

2. **法院层级**: 修改点击的链接文本
   - 最高法院
   - 高级法院
   - 中级法院
   - 基层法院

3. **检索关键词**: 修改 `search_keyword` 变量

4. **排序方式**: 取消注释相应代码
   - 按裁判日期排序
   - 按审判程序排序

### 注意事项

- 下载速度不要太快，避免被封 IP
- 建议分批次下载，每次不超过 100 个文档
- 定期检查下载目录，确保文件正常保存
- 遵守网站使用规则，合理使用爬虫

### 常见问题

1. **元素找不到**: 网站可能更新了结构，需要重新定位元素
2. **登录失败**: 检查账号密码是否正确，是否需要验证码
3. **下载失败**: 检查下载路径是否存在，是否有写入权限
4. **速度太慢**: 可以适当减少 sleep 时间，但要注意反爬虫